# Temporal Feature Engineering: Lags & Windows

In Lesson 06, we engineered our Feature Matrix ($X$) by looking at the *calendar*. We extracted the month, the day of the week, and the distance to the nearest holiday.

But if we are predicting tomorrow's stock price, knowing that tomorrow is a "Tuesday in November" is not enough. The most important predictor of tomorrow's price is *today's price*.

To unleash powerful Machine Learning algorithms (like XGBoost or Random Forests) on Time Series data, we must teach them how to "remember" the past. We do this by physically converting the 1D chronological timeline into a 2D tabular matrix using **Lag** and **Window** features.

In classical statistics (ARIMA), the algorithm handles the past automatically. In Machine Learning, the algorithm is time-blind. It evaluates every row in your database as an isolated, independent universe. Our goal is to forcefully inject the history of the target variable ($y$) directly into the feature columns ($X$) of the current row.

Let's set up our Python environment to slice the timeline into a matrix.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Autoregressive Matrix Engineering Environment Ready.")

✅ Autoregressive Matrix Engineering Environment Ready.


# 1. Lag Features (The Mathematical Memory)

A **Lag** is simply the value of the target variable from a prior time step.
If we want to predict today ($y_t$), we can use yesterday ($y_{t-1}$) and the day before yesterday ($y_{t-2}$) as input features.

Mathematically, we are transforming a sequence:


$$[y_1, y_2, y_3, y_4, y_5]$$


Into a supervised learning matrix where $X$ maps to $y$:


$$X_t = [y_{t-1}, y_{t-2}, \dots, y_{t-k}] \implies \text{Predict } y_t$$

In `pandas`, we achieve this using the `.shift()` method.

In [2]:
# 1. Simulate a simple Daily Sales dataset
dates = pd.date_range(start='2023-01-01', periods=10, freq='D')
sales = [100, 105, 112, 108, 120, 130, 125, 140, 135, 150]
df = pd.DataFrame({'Sales': sales}, index=dates)

# 2. Engineer Lag Features (Shift the data forward in time)
df['Lag_1'] = df['Sales'].shift(1) # Yesterday's Sales
df['Lag_2'] = df['Sales'].shift(2) # Day Before Yesterday's Sales
df['Lag_7'] = df['Sales'].shift(7) # Sales from exactly one week ago

print("🚨 Notice the NaNs! We cannot know the 'yesterday' of the very first day.")
display(df.head(5))

🚨 Notice the NaNs! We cannot know the 'yesterday' of the very first day.


,Sales,Lag_1,Lag_2,Lag_7
2023-01-01,100,NaN,NaN,NaN
2023-01-02,105,100.0,NaN,NaN
2023-01-03,112,105.0,100.0,NaN
2023-01-04,108,112.0,105.0,NaN
2023-01-05,120,108.0,112.0,NaN


# 2. Rolling Window Features (Capturing Momentum)

Lags are highly susceptible to sudden noise. If yesterday had a massive, random spike in sales, `Lag_1` will feed that extreme anomaly directly into the model, potentially throwing off the prediction.

To provide the algorithm with a smoother understanding of recent history, we engineer **Rolling Window Features**. Instead of looking at a single day, we calculate summary statistics (Mean, Max, Min, Volatility) over a sliding block of time $w$.

$$RollingMean_t = \frac{1}{w} \sum_{i=1}^{w} y_{t-i}$$

### ⚠️ The Data Leakage Trap (CRITICAL)

When calculating a 7-day rolling mean to predict Day 8, **you must not include Day 8 in the calculation.** If you run a standard `df.rolling(7).mean()`, Pandas will include the current row ($y_t$) in the average. You are feeding the answer ($y_t$) into the features ($X_t$). Your model will achieve $99\%$ accuracy in training and fail instantly in production.

**The Golden Rule of ML Forecasting:** You must *always* `.shift(1)` before applying `.rolling()`.

In [3]:
# 1. Create a 3-Day Rolling Mean (Safely shifted to prevent leakage!)
df['Rolling_3D_Mean'] = df['Sales'].shift(1).rolling(window=3).mean()

# 2. Create a 3-Day Rolling Standard Deviation (Volatility)
df['Rolling_3D_Std'] = df['Sales'].shift(1).rolling(window=3).std()

# 3. Create a Rolling Maximum (What was the highest peak recently?)
df['Rolling_3D_Max'] = df['Sales'].shift(1).rolling(window=3).max()

print("🚨 Leak-Proof Rolling Features Constructed.")
display(df[['Sales', 'Lag_1', 'Rolling_3D_Mean', 'Rolling_3D_Max']].tail(5))

🚨 Leak-Proof Rolling Features Constructed.


,Sales,Lag_1,Rolling_3D_Mean,Rolling_3D_Max
2023-01-06,130,120.0,113.333333,120.0
2023-01-07,125,130.0,119.333333,130.0
2023-01-08,140,125.0,125.000000,130.0
2023-01-09,135,140.0,131.666667,140.0
2023-01-10,150,135.0,133.333333,140.0


# 3. Expanding Window Features (Long-Term State)

While rolling windows have a fixed size (e.g., looking at the past 7 days), an **Expanding Window** looks at *everything* from the very beginning of the dataset up until yesterday.

This is crucial for calculating cumulative metrics like:

* "What is the historical average of this customer's spend over their entire lifetime?"
* "What is the all-time high temperature recorded for this machine?"

In [ ]:
# Create an Expanding Mean (Lifetime Average), safely shifted!
df['Lifetime_Avg'] = df['Sales'].shift(1).expanding().mean()

# 4. Building the Ultimate ML Matrix (Tabularization)

Let's put it all together on a larger simulated dataset. We will engineer the features, and then perform the final, mandatory step: **Dropping the NaNs**.

Because shifting data inherently creates missing values at the top of the dataset, we must slice off the beginning of our matrix before feeding it to an algorithm.

In [5]:
# 1. Simulate a larger dataset
np.random.seed(42)
dates_large = pd.date_range(start='2023-01-01', periods=100, freq='D')
sales_large = np.linspace(100, 500, 100) + np.sin(np.arange(100) * 0.5) * 50 + np.random.normal(0, 10, 100)
ml_df = pd.DataFrame({'Target_Sales_Today': sales_large}, index=dates_large)

# 2. Engineer Lags
for i in [1, 2, 3, 7]:
    ml_df[f'Lag_{i}'] = ml_df['Target_Sales_Today'].shift(i)

# 3. Engineer Leak-Proof Windows
ml_df['Roll_7D_Mean'] = ml_df['Target_Sales_Today'].shift(1).rolling(7).mean()
ml_df['Roll_7D_Std'] = ml_df['Target_Sales_Today'].shift(1).rolling(7).std()

# 4. The Final Cut (Dropping NaNs)
# The maximum lookback we used was 7 days (Lag_7 and Roll_7D). 
# Therefore, the first 7 rows are mathematically corrupted with NaNs. We drop them.
print(f"Rows before dropping NaNs: {len(ml_df)}")
ml_df_clean = ml_df.dropna()
print(f"Rows after dropping NaNs:  {len(ml_df_clean)}")

# 5. Extract X and y for XGBoost/Random Forest!
X = ml_df_clean.drop(columns=['Target_Sales_Today'])
y = ml_df_clean['Target_Sales_Today']

print("\n✅ Matrix successfully tabularized. Ready for Supervised Machine Learning!")

Rows before dropping NaNs: 100
Rows after dropping NaNs:  93

✅ Matrix successfully tabularized. Ready for Supervised Machine Learning!


## Real-World Use Case or Analogy:

Think of Lag and Window Features like a **Stock Day Trader looking at their monitors**:

* **The Target ($y_t$)**: The price of Apple stock *right now*. This is what they want to predict.
* **Lag Features**: The trader glances at the ticker tape. "What was the exact price 1 hour ago? What was it 2 hours ago?" (Lags 1 and 2).
* **Rolling Features**: The trader looks at the MACD or Moving Average chart. "What is the average price over the last 14 hours? Is the volatility (Std Dev) shrinking or exploding over the last 5 hours?"
* **Tabularization**: The trader takes all of those historical indicators, writes them down on a single horizontal row on a piece of paper, and uses that single row of context to make a prediction for the next minute.

---